In [1]:
from langgraph.graph import StateGraph ,START, END
from typing import TypedDict, Annotated
from langchain_core.messages import BaseMessage, HumanMessage
from langchain_groq import ChatGroq
from langgraph.graph.message import add_messages
from dotenv import load_dotenv

from langgraph.prebuilt import ToolNode, tools_condition
from langchain_community.tools import DuckDuckGoSearchRun
from langchain_core.tools import tool

import requests
import random

C:\Users\Krishna gupta\AppData\Local\Temp\ipykernel_32620\2109649532.py:9: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.tools import DuckDuckGoSearchRun


In [3]:
load_dotenv()

llm = ChatGroq(
    model="openai/gpt-oss-120b",
    temperature=0.7
)

In [11]:
# Tools
search_tool = DuckDuckGoSearchRun(region='us-en')

@tool
def calculator(first_num:float, second_num:float, operation:str)-> dict:
  """
  Perform a basic arithmetic operation on two numbers.
  Supported operations: add, sub, mul, div
  """
  try:
    if operation == "add":
      result = first_num + second_num
    elif operation == "sub":
      result = first_num - second_num
    elif operation == "mul":
          result = first_num * second_num
    elif operation == "div":
          result = first_num / second_num
    else:
       return {"error":f"Unsupported opearation '{operation}'"}
    return {"first_num":first_num,"second_num":second_num,"operation":operation,"result":result}
  except Exception as e:
     return {"error":str(e)}

@tool
def get_stock_price(symbol: str) -> dict:
    """
    Fetch intraday stock data for a given symbol.
    Examples: AAPL, TSLA, IBM.
    """

    url = "https://www.alphavantage.co/query"

    params = {
        "function": "TIME_SERIES_INTRADAY",
        "symbol": symbol.upper(),
        "interval": "5min",
        "apikey": "CUSRY2BWCITFJC7M"
    }

    try:
        response = requests.get(url, params=params, timeout=15)
        response.raise_for_status()

        return response.json()

    except requests.RequestException as e:
        return {"error": str(e)}

  

     

In [13]:
# Make tool List
tools = [get_stock_price,search_tool,calculator]

# Make the LLM tool-aware
llm_with_tools = llm.bind_tools(tools)

In [6]:
# state
class ChatState(TypedDict):
  messages:Annotated[list[BaseMessage],add_messages]
  

In [14]:
# Graph node
def chat_node(state:ChatState):
  """LLM node that may answer or request a tool call."""
  messages = state['messages']
  response = llm_with_tools.invoke(messages)
  return {'messages':[response]}

tool_node = ToolNode(tools) # Executes tool calls


In [ ]:
# graph structure
graph = StateGraph(ChatState)
